# Hierarchical Clustering and Dendrograms on METABRIC Data
*Authored by Dr. Noelle Anderson in 2026*

In Notebook 2B, you used K-means clustering and thought carefully about how many clusters to use. In this notebook, you will learn a different clustering method called **hierarchical clustering**.

Hierarchical clustering does not begin by choosing one fixed number of clusters. Instead, it starts with very small groups and builds upward by merging the most similar samples or groups step by step. In this notebook, we will use **agglomerative hierarchical clustering**, which means we begin with each sample on its own and then merge upward into larger groups. The result is a tree-like structure called a **dendrogram**.

This matters in biology, biochemistry, and public health because real datasets often contain broad patterns with smaller subgroups nested inside them. For example, in biochemistry, a set of enzyme variants might separate into larger groups based on overall activity and then into smaller subgroups based on binding strength or response to pH. In public health, communities may cluster broadly by overall environmental and health burden, while still containing smaller subgroups shaped by different mixes of pollution exposure, housing conditions, income, or access to care. Hierarchical clustering helps us explore that layered structure without assuming from the start that there is only one useful level of grouping.

By the end of this notebook, you should be able to:
- explain hierarchical clustering in plain language
- compute pairwise distances and build a hierarchical tree with SciPy
- read the main parts of a dendrogram
- explain the difference between truncating a dendrogram and cutting it into flat clusters
- use the same hierarchy to create broader and finer clusterings on METABRIC

## Notebook roadmap

In this notebook, you will:

1. Build a tiny simulated toy dataset
2. Use distances to see why nearby points tend to merge early
3. Build and read a toy dendrogram
4. Load the scaled METABRIC dataset from Google Drive
5. Build a hierarchical clustering tree for METABRIC with Ward linkage
6. Use truncated dendrograms to inspect the structure more readably
7. Create broader and finer flat clusterings from the same hierarchy
8. Interpret what hierarchical clustering is useful for and what its limits are

## Step 0: Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import silhouette_score # from last time

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster # new!
from scipy.spatial.distance import pdist, squareform # new!

*Note: This week we are using the package **`SciPy`** instead of `sklearn` because SciPy gives us tools that work naturally with hierarchical clustering trees, especially `linkage`, `dendrogram`, and `fcluster`. That makes it much easier to build the hierarchy, visualize it, and then turn it into flat clusters.*


## Step 1: Build and plot a tiny toy dataset

Before we work with METABRIC, we will start with a very small dataset that we can inspect point by point.

This toy dataset is not meant to be realistic biomedical data, its job is to help us see what hierarchical clustering is doing on an easy-to-visualized 2D dataset before we move to a much larger real dataset.

In [ ]:
# Create a tiny simulated dataset
# The sample names will help us read the dendrogram later

toy_df = pd.DataFrame(
    {
        "sample": ["A", "B", "C", "D", "E", "F"],
        "x": [1.0, 1.4, 2.6, 6.0, 6.3, 7.4],
        "y": [1.0, 1.3, 2.4, 6.2, 5.8, 7.1],
    }
).set_index("sample")

print("Toy dataset:")
toy_df

Let's make a scatterplot of the toy dataset using `x` on the x-axis and `y` on the y-axis. We'll add a text label for each sample so the points are easy to identify. We wouldn't be able to visualize data with more than 2 features in real feature space since plots with more than 2 dimensions are often hard to interpret.

In [ ]:
# Make a scatterplot of the toy data
plt.figure(figsize=(6, 6))
plt.scatter(toy_df["x"], toy_df["y"])

# Loop through each sample name in the dataframe index
# Here, i will be samples "A", "B", "C", and so on
for i in toy_df.index:
    x_value = toy_df.loc[i, "x"] # Look up the x-coordinate for this sample
    y_value = toy_df.loc[i, "y"] # Look up the y-coordinate for this sample

    # Add the sample name as text near the point with plt.text
    # We add 0.05 to each coordinate so the label does not sit directly on top of the dot
    plt.text(x_value + 0.05, y_value + 0.05, i)

# Add a title and axis labels
plt.title("Toy Dataset for Hierarchical Clustering")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

When we talk about **distance** here, like with K-means, we just mean a numerical way to describe how far apart two samples are. In clustering, smaller distances usually mean samples are more similar, and larger distances mean they are more different.

For this toy dataset, you can begin by judging distance visually from the scatterplot. Points that are very close together may merge earlier, while points that are farther apart may merge later.

### <font color=0D9488>**Question 1:**</font> Based on the scatterplot alone and what you read in the introduction about how agglomerative clustering works, which pair of points do you think will merge first in the hierarchy? Briefly explain your reasoning in plain language.

**Your solution here!**

## Step 2: Turn visual closeness into a distance matrix

So far, you judged closeness by eye. Now we will compute those distances directly.

In hierarchical clustering, we often start by measuring the distance between **every pair of samples**. These are called **pairwise distances**. If two samples have a small pairwise distance, they are relatively similar. If they have a large pairwise distance, they are relatively different.

In this notebook, we will use **Euclidean distance**, which is the ordinary straight-line distance between points. We will use `pdist(...)` to compute all pairwise distances, and then `squareform(...)` to turn that result into a readable square distance matrix.


In [ ]:
# Compute all pairwise distances between toy samples
# pdist(...) returns the distances in condensed form
# squareform(...) turns them into a square distance matrix

toy_distances = pdist(toy_df[["x", "y"]], metric="euclidean")

toy_distance_matrix = pd.DataFrame(
    squareform(toy_distances),
    index=toy_df.index,
    columns=toy_df.index,
)

print("Toy distance matrix, rounded to 2 decimals:")
display(toy_distance_matrix.round(2))

Notice that the distance matrix is **symmetric**: the distance from `A` to `B` is the same as the distance from `B` to `A`. The diagonal is all 0s because each sample has distance 0 from itself.

This matrix gives us a concrete way to see which points are closest together before we build the hierarchy.


### <font color=0D9488>**Question 2:**</font> Find the smallest off-diagonal (non-0) value in the distance matrix. Which two samples does it connect, are they the same you identified visually in the last question? Why does that matter for hierarchical clustering?

**Your solution here!**

## Step 3: Build and read the toy hierarchy

Now that we have a way to measure how far apart samples are, we need a rule for deciding how groups should merge step by step into clusters. That rule is called the **linkage method**.

In this notebook, we will use **Ward linkage**. Ward linkage tends to merge points or groups in a way that keeps clusters relatively compact and similar in spread. Other linkage methods exist, such as complete, average, and single linkage, and different choices can produce different trees. For this first notebook on hierarchical clustering, we will stick to one clear starting point.

SciPy's `linkage(...)` function uses the feature values together with the linkage rule to decide which samples or groups merge first, second, and so on, until the full hierarchy is built.

Important coding details:
- We pass in the numeric feature columns only. Here that means `x` and `y`.
- We set `method="ward"` to use Ward linkage.
- The result is called a **linkage matrix**.


In [ ]:
# Build the hierarchical clustering tree for the toy data
Z_toy = linkage(toy_df[["x", "y"]], method="ward")

SciPy stores the merge steps in a **linkage matrix**, here called `Z_toy`. This is not the same thing as the distance matrix from Step 2.

- The **distance matrix** told us how far apart every pair of samples was.
- The **linkage matrix** stores the sequence of merges that builds the hierarchy.

We will focus on the dendrogram rather than the linkage matrix itself, because the tree is much easier to interpret visually.


We will now use `dendrogram(...)` to draw the toy hierarchy and label the leaves with the sample names.

A **dendrogram** is a tree diagram showing how samples or groups merge step by step. The samples start as the leaves at the bottom, and the tree is read from the bottom upward. The vertical position where two branches join is called the **merge height**. The merge height reflects how different the two groups were when they were combined:
- a **low merge height** means the samples or groups were very similar and merged early
- a **high merge height** means they were more different and only merged later

So the height of each connection tells you how similar or dissimilar those groups were at the moment they joined. The branch colors are not the main thing to interpret here, they are just a plotting aid based on a threshold.


In [ ]:
plt.figure(figsize=(10, 5))
dendrogram(Z_toy, labels=toy_df.index)
plt.title("Toy Dendrogram")
plt.xlabel("Samples")
plt.ylabel("Merge height")
plt.show()

### <font color=0D9488>**Question 3:**</font> Look at the toy dendrogram. Which merges happen at relatively low heights and does this match what you saw earlier in your 2D scatterplot and in your distance matrix from `pdist`?

**Your solution here!**

With a tiny dataset, the full dendrogram is readable. With larger datasets, the full tree can become crowded and hard to interpret.

That is why **truncation** is useful. It lets us simplify the **plot** so we can focus on the larger, higher-level merges without being overwhelmed by all the small, early merges near the bottom.

Truncating the plot does **not** change the hierarchy itself. It only changes how much of the tree we display at once. Later, when we use `fcluster(...)`, we will be **cutting** the hierarchy to create flat clusters. That changes the grouping.

We will use `truncate_mode="lastp"` with `dendrogram(...)`, which asks SciPy to show only the last `p` merged clusters. Looking at more than one value of `p` can make the tree easier to read.


In [ ]:
plt.figure(figsize=(5, 3))
dendrogram(Z_toy, labels=toy_df.index, truncate_mode="lastp", p=3)
plt.title("Toy Dendrogram Truncated to the Last 3 Merged Clusters")
plt.xlabel("Truncated branch structure")
plt.ylabel("Merge height")
plt.show()

plt.figure(figsize=(5, 3))
dendrogram(Z_toy, labels=toy_df.index, truncate_mode="lastp", p=5)
plt.title("Toy Dendrogram Truncated to the Last 5 Merged Clusters")
plt.xlabel("Truncated branch structure")
plt.ylabel("Merge height")
plt.show()

When we move from `p=3` to `p=5`, we can see more of the smaller branches instead of only the broadest summary structure. Looking at more than one truncation level can help because one view may be too compressed, while another may reveal enough detail to make the merge pattern easier to interpret.

The numbers at the tips of a truncated branch show how many original samples are being grouped together inside that collapsed leaf.


## Step 4: Making flat clusters from hierarchical ones

So far, we have built a hierarchical clustering tree. Now we want to turn it into **flat clusters**, where each sample is assigned to one group.

One way to do that is to decide how many clusters we want, just like we did with K-means. In SciPy, we will use `fcluster(...)` with `criterion="maxclust"`. That option tells SciPy to return exactly the number of flat clusters we ask for.

A common practical way to choose a cut is to look for a **larger vertical gap** in the dendrogram and cut below that jump. This is similar in spirit to how, in Notebook 2B, you used visual evidence and silhouette score to judge a reasonable value of `k` for K-means. Here we are using the dendrogram visually instead and will use silhouette score later on when we work with the real METABRIC data.

For example:
- What if we want **2 large clusters**?
- What if we want **4 smaller clusters**?

We can do this by cutting the tree so that it produces a chosen number of groups.


We will now create **2 clusters** from the hierarchical tree, because in our dendrogram it looked like there were about 2 main groups at a similar merge height.

We will:
- Use `fcluster(...)` with `criterion="maxclust"` so SciPy returns the number of clusters we request
- Set `t=2` to ask for 2 clusters
- Store the result in `toy_clusters_2`
- Add it to the dataframe as `"2-clusters"`
- Print how many clusters were created and how many samples are in each cluster


In [ ]:
# Cut the tree at 2 clusters
toy_clusters_2 = fcluster(Z_toy, t=2, criterion="maxclust")

toy_df["2-clusters"] = toy_clusters_2

print("Number of clusters:", toy_df["2-clusters"].nunique())
print("Cluster counts:")
print(toy_df["2-clusters"].value_counts().sort_index())

toy_df

### <font color=0D9488>**Question 4:**</font> Now create **4 clusters** from the same tree.

In [ ]:
# Your solution here!

Notice how the same hierarchy can be turned into different flat clusterings. There is often not one single "right" answer, the choice depends on the method, the data, and the level of detail that is useful for your question.

Remember:
- **Truncating** in `dendrogram` only changes how much of the tree we *see* on the plot. It does not change the clustering.
- **Cutting** with `fcluster` uses the tree to actually assign cluster labels. It *does* change how the data are grouped.

So truncation is about visualization, while cutting is about creating clusters.


## Part 2: Step 1: Move to the METABRIC dataset

Now that the basic ideas are clearer on a small dataset, we will apply the same workflow to our more complex METABRIC dataset. The main steps are still the same: build the hierarchy, inspect the dendrogram, and then choose a flat clustering from that tree. The main difference is that this dataset is much larger, so we will rely on truncation much more heavily to make the dendrogram readable.

We will use the same scaled dataset from last week, `metabric_1b_scaled.csv`. The rows are patient samples, and the columns are numeric features. The `patient_id` column should be treated as the index, not as a feature.

One important limitation to keep in mind is that clustering depends on how we define similarity. Here, we are using Ward linkage on scaled numeric features, so samples are grouped based on overall numerical closeness across those features. In some scientific settings that is useful. In others, a researcher may care more about whether profiles rise and fall in similar patterns rather than whether their values are close in size. That is one reason clustering results should always be interpreted in context.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Load the scaled METABRIC data from Google Drive
file_path = "/content/drive/MyDrive/SCIP2026ML/metabric_1b_scaled.csv"

# patient_id should be used as the index, not as a feature
metabric_df = pd.read_csv(file_path, index_col="patient_id")

display(metabric_df.head())

## Part 2: Step 2: Build the METABRIC hierarchy and inspect it with truncation

We will again use `linkage(...)` with `method="ward"`.

The hierarchy is built from the full scaled dataset. The dendrograms below are **truncated views** of that full hierarchy so the structure is easier to read. In other words, truncation helps with visualization, but the underlying tree is still the full one. For a large dataset like this, a full dendrogram is usually too crowded to be useful for interpretation.


### <font color=0D9488>**Question 5:**</font> Use the `linkage` command to create a **linkage matrix** called `Z_metabric` for the full `metabric_df` using Ward's distance.


In [ ]:
# Your solution here!

Rather than plotting the full dendrogram, which would be extremely crowded and not very readable, we will go straight to **truncated views** of the same full hierarchy. Looking at more than one truncation level can help us see both the broadest structure and a bit more of the smaller branches underneath it.


### <font color=0D9488>**Question 6:**</font> Make two truncated dendrograms of the same full METABRIC hierarchy so you can compare how the amount of truncation changes what you see.

Your plots should both use `Z_metabric`, but use different values of `p`:
- one plot with `truncate_mode="lastp"` and `p=5`
- one plot with `truncate_mode="lastp"` and `p=20`

For each plot:
- use `figsize=(12, 5)`
- include a title
- label the x-axis as `"Truncated branch structure"`
- label the y-axis as `"Merge height"`


In [ ]:
# Your solution here!

These plots are still based on the full hierarchy, but they are much easier to read than the full tree would be. For large datasets, some visual messiness is expected, so truncation is not a shortcut or a mistake. It is a practical way to focus on the larger merge structure.


### <font color=0D9488>**Question 7:**</font> Based on the truncated METABRIC dendrograms, how many flat clusters would you assign this data to at this stage? Briefly explain your answer using the merge heights and overall branch structure.


**Your solution here!**

## Part 2: Step 3: Create flat clusterings for METABRIC


### <font color=0D9488>**Question 8:**</font> Use your answer from the last question as a starting point and create **two** flat clusterings from the same hierarchy: one with `t=3` and one with `t=4`, using `criterion="maxclust"` in `fcluster()`. Add both sets of cluster labels to `metabric_df` as new columns named `cluster_3` and `cluster_4`. Then print the value counts for both columns as a final check.


In [ ]:
# Your solution here!

We want to look out for overly small clusters. If less than about 5% of the data falls into a cluster, it could be noise, an outlier group, or a result of over-splitting rather than a meaningful pattern. This is only a practical starting point, not a universal rule, but it can help you notice when a clustering may be getting too fragmented.


## Part 2: Step 4: Use silhouette score as a second check

In Notebook 2B, you used **silhouette score** with K-means. We can also use it here after we turn the hierarchy into flat clusters. The idea here is the same: a **higher silhouette score** usually means points are closer to others in their own cluster and farther from points in other clusters. It does not prove that one choice is "correct," but it gives us one more piece of evidence.

Here, we will compare the silhouette scores for the `t=3` and `t=4` clusterings we just created.


In [ ]:
# Compute silhouette scores for the two hierarchical clustering choices

# silhouette_score() needs TWO separate inputs:
# 1. the original feature data only; 2. the cluster labels for each sample

# metabric_df currently includes both:
# - the scaled METABRIC feature columns
# - the added cluster label columns: "cluster_3" and "cluster_4"

# So first, we will remove the cluster label columns to get back only the feature data
X_features = metabric_df.drop(columns=["cluster_3", "cluster_4"])

# Now compute the silhouette score for the t = 3 clustering
# X_features = the original scaled data
# metabric_df["cluster_3"] = the cluster label assigned to each sample
silhouette_3 = silhouette_score(X_features, metabric_df["cluster_3"])

# Compute the silhouette score for the t = 4 clustering
silhouette_4 = silhouette_score(X_features, metabric_df["cluster_4"])

# Print both scores rounded to 3 decimal places
print(f"Silhouette score for t = 3: {silhouette_3:.3f}")
print(f"Silhouette score for t = 4: {silhouette_4:.3f}")

### <font color=0D9488>**Question 9:**</font> Use the dendrogram views, the cluster counts, and the silhouette scores together. At this stage, would you choose `t=3` or `t=4` for METABRIC? Briefly explain your choice using at least two pieces of evidence.


**Your solution here!**


## Part 2: Step 5: Interpret what hierarchical clustering is useful for

So far in this notebook, you have:
- built a hierarchical clustering tree step by step
- used dendrograms to visualize how samples merge at different levels
- explored how truncation changes what you can see in the tree
- created flat clusters by cutting the tree in different ways
- compared broader versus finer groupings of the same data

This is a different way of thinking about clustering than K-means.

In **K-means**, we:
- choose the number of clusters `k` first
- then assign each sample to one of those clusters

In **hierarchical clustering**, we:
- build the full structure of how samples group together
- then decide how to cut that structure depending on how much detail we want

Because of this, hierarchical clustering is especially useful when:
- we want to explore both **broad groups and smaller subgroups** in the same framework
- we are **not sure how many clusters** are appropriate ahead of time
- we care about whether the data might have a **nested or layered structure**

However, it also has some important limits:
- large dendrograms can become difficult to read and interpret
- the results depend on how we define similarity, including distance and linkage choices
- clusters in biomedical data do not automatically correspond to true biological subtypes, so interpretation should be done carefully

In practice, hierarchical clustering is often used as an **exploratory tool** to understand structure in the data, while methods like K-means can be useful when we want a specific number of clusters for downstream analysis. The methods can support each other rather than compete: hierarchical clustering can suggest a reasonable range of cluster counts, and K-means can then be used when we want one particular flat clustering to compare more directly.


# Congratulations, you have completed today's notebook!

## Key Takeaways:
- You used **SciPy** for hierarchical clustering because it works naturally with `linkage`, `dendrogram`, and `fcluster`.
- You learned that hierarchical clustering builds a tree of merges instead of choosing all flat clusters at once.
- You used a distance matrix, a dendrogram, and flat cluster labels to follow the same hierarchy from several angles.
- You saw that **truncation** changes the display of the tree, not the underlying hierarchy.
- You created flat clusterings from the same hierarchical tree on METABRIC.
- You used silhouette score as a second check after turning the hierarchy into flat clusters.
- You practiced interpreting what hierarchical clustering can show us and what it cannot prove.

In this notebook, you followed an unsupervised learning workflow adapted for hierarchical clustering: load data, inspect similarity, build the hierarchy, visualize the tree, create flat clusters, and interpret the result carefully. The central new idea was that hierarchical clustering keeps the merge history, so we can examine structure from the same tree instead of starting with only one flat answer.

## Where This Fits in the ML Workflow
In this notebook, you explored another way to find structure in unlabeled data. Instead of starting by choosing `k`, you built a hierarchy first and then decided how broadly or finely to cut it. That makes hierarchical clustering a useful exploratory tool when we want to examine nested structure and compare multiple possible flat clusterings.

In the next notebook, you will compare hierarchical clustering with K-means more directly and think about how different clustering methods can produce different groupings on the same data.
